# 03 — Predictive Modelling
Late-delivery risk classification and demand forecasting.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.features import add_delivery_features, add_inventory_features
from src.models import train_late_delivery_model, train_demand_model

sns.set_theme(style='whitegrid')
%matplotlib inline

PROCESSED = Path('..') / 'data' / 'processed'
orders = pd.read_csv(PROCESSED / 'orders_clean.csv', parse_dates=['order_date','promised_date','actual_delivery_date'])
inventory = pd.read_csv(PROCESSED / 'inventory_clean.csv', parse_dates=['date'])


## 3.1 Feature engineering for models

In [ ]:
orders = add_delivery_features(orders)
inventory = add_inventory_features(inventory)
print("Orders feature columns:", orders.columns.tolist())


## 3.2 Late-delivery classification

In [ ]:
available_features = [c for c in ['distance_km','stop_count','loading_time_hours','congestion_level','vehicle_capacity','order_quantity','day_of_week','is_weekend_order'] if c in orders.columns]
print("Using features:", available_features)
if available_features and 'late_delivery_flag' in orders.columns:
    model, metrics = train_late_delivery_model(orders, features=available_features)
    print("\nMetrics:", metrics)
else:
    print("Skipped: required columns not present in dataset.")
    print("Available:", orders.columns.tolist())


## 3.3 Feature importances

In [ ]:
try:
    rf = model.named_steps['classifier']
    pre = model.named_steps['preprocessor']
    feat_names = pre.get_feature_names_out()
    importances = rf.feature_importances_
    fi_df = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9,5))
    sns.barplot(data=fi_df, y='feature', x='importance', ax=ax, palette='viridis')
    ax.set_title('Feature Importances — Late Delivery Model')
    plt.tight_layout()
    plt.savefig('../outputs/figures/feature_importances.png', dpi=150)
    plt.show()
except Exception as e:
    print(f"Skipped feature importance plot: {e}")


## 3.4 Demand forecasting

In [ ]:
demand_features = [c for c in ['rolling_demand_7d','demand_cv','days_of_cover','month','week_number'] if c in inventory.columns]
print("Using demand features:", demand_features)
if demand_features and 'demand' in inventory.columns:
    d_model, d_metrics = train_demand_model(inventory, features=demand_features)
    print("\nDemand model metrics:", d_metrics)
else:
    print("Skipped: required columns not present in dataset.")
    print("Available:", inventory.columns.tolist())
